## Hacemos comparativa con diferentes Hiperparámetro

### Usaremos RAG + DSLR


In [ ]:
### Cargamos el modelo phi-4
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "microsoft/phi-4-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_lm = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")
generator = pipeline("text-generation", model=model_lm, tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.84it/s]
Device set to use cuda:0


In [12]:
import pickle
#Usaremos los embedding y metadatos usados anteriormente
with open(f"../outputs/embeddings_gemma_y_metadatos.pkl", "rb") as f:
    data = pickle.load(f)

embeddings = data["embeddings"]
metadatos = data["metadatos"]

In [13]:
#Uso FAISS para indexar
#FAISS es una librería desarrollada por Meta (Facebook) para hacer búsqueda rápida de vectores por similitud, ideal cuando tienes muchos embeddings (como en RAG).
import faiss
import numpy as np
embedding_matrix = np.array(embeddings)

# Crear índice FAISS (búsqueda por similitud L2 o Euclidiana)
dim = embedding_matrix.shape[1] 
print(dim)
index = faiss.IndexFlatL2(dim)  

# Agregar los vectores al índice
index.add(embedding_matrix)

# Guardar el índice en disco
faiss.write_index(index, "../outputs/faiss_index.index")

768


In [14]:
import pandas as pd
# Cargar el archivo CSV
df_qa = pd.read_csv("../dataQA/qa.txt")
df_qa.head()

,chunk,question,answer
0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...
1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...
2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...
3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...
4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can..."


### DSLR

In [15]:
import spacy
nlp = spacy.load("en_core_web_sm")

def dividir_oraciones(texto):
    doc = nlp(texto)
    return [sent.text.strip() for sent in doc.sents]

In [16]:
from sentence_transformers import CrossEncoder


def re_rank_oraciones(question, oraciones):
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [(question, sent) for sent in oraciones]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(oraciones, scores), key=lambda x: x[1], reverse=True)
    return ranked

In [17]:
import numpy as np

def filtrar_por_umbral(oraciones_ranked, percentil=90):
    scores = [score for _, score in oraciones_ranked]
    umbral = np.percentile(scores, percentil)
    oraciones_filtradas = [(sent, score) for sent, score in oraciones_ranked if score >= umbral]
    return oraciones_filtradas, umbral

In [18]:
def reconstruir_contexto(oraciones_filtradas, oraciones_originales):
    oraciones_validas = set([sent for sent, _ in oraciones_filtradas])
    reconstruido = [sent for sent in oraciones_originales if sent in oraciones_validas]
    return reconstruido

In [19]:
## Funcion para implementar los 3 pasos de DSLR 
def refine_documents(sentence,pregunta):
    # Paso 1: Separar oraciones
    oraciones = dividir_oraciones(sentence)

    # Paso 2: Rankear oraciones
    oraciones_ranked = re_rank_oraciones(pregunta, oraciones)

    # Paso 3: Filtrar con percentil 90
    oraciones_filtradas, umbral = filtrar_por_umbral(oraciones_ranked, percentil=90)

    # Paso 4: Reconstruir en orden original
    oraciones_reconstruidas = reconstruir_contexto(oraciones_filtradas, oraciones)

    # Resultado final para pasar al LLM
    documento_refinado = " ".join(oraciones_reconstruidas)
    return documento_refinado
         

In [ ]:
import torch
## Implementamos DSLR en nuestro pipeline
def responder_con_phi4_con_contexto_refinado(preguntas, modelo_embedding,temperature=0.5, k=5, inEnglish= False,):
    resultados = []
    idioma = "La respuesta tiene que ser obligatoriamente en ingles" if inEnglish else ""
     # Embeddear la pregunta
    preguntas_vec = modelo_embedding.encode(preguntas, convert_to_tensor=True)

    # Buscar k chunks relevantes en FAISS
    D, I = index.search(preguntas_vec.cpu().numpy(), k)

    for idx_preg, pregunta in enumerate(preguntas):
        # Recuperar contexto relevante
        chunks_usados = []
        contexto = ""
        for idx in I[idx_preg]:
            doc = metadatos[idx]
            chunk_text = doc["chunk"].strip()
            titulo = doc.get("id_doc", "Sin título")

            # Refinar con DSLR
            texto_refinado = refine_documents(chunk_text, pregunta)
            chunks_usados.append({"titulo": titulo, "chunk": texto_refinado})
            contexto += f"- {texto_refinado}\n"

        # Construir prompt completo
        prompt = (
            f"<|user|>\nUsa el siguiente contexto para responder la pregunta de manera clara y precisa.\n\n"
            f"Contexto:\n{contexto}\nPregunta: {pregunta} {idioma}\n<|assistant|>"
        )

        #  Generar respuesta con Phi-4
        with torch.no_grad():
            output = generator(
                prompt,
                max_new_tokens=300,
                temperature=temperature,
                do_sample=True
            )[0]["generated_text"]

        respuesta = output[len(prompt):].strip()

        resultados.append({
            "pregunta": pregunta,
            "respuesta": respuesta,
            "chunks_usados": chunks_usados
        })

    return resultados

In [32]:
from sentence_transformers import SentenceTransformer
hiperparameters_grid = [0.1, 0.3]
batch_size = 16  
preguntas = df_qa.head(100)["question"].tolist()
modelo_embedding = SentenceTransformer("google/embeddinggemma-300m")
for temperature in hiperparameters_grid:
    print(f"\n Procesando con temperature={temperature}")
    for start in range(0, len(preguntas), batch_size):
        batch = preguntas[start:start + batch_size]

        resultados_batch = responder_con_phi4_con_contexto_refinado(
            batch,
            modelo_embedding,
            temperature=temperature,
            k=5,
            inEnglish=True
        )

        # Guardar resultados en df_qa
        for i, resultado in enumerate(resultados_batch):
            idx = start + i
            df_qa.at[idx, f"answer_modelo_rag_dslr_h_{temperature}"] = resultado["respuesta"]

    torch.cuda.empty_cache()  


 Procesando con temperature=0.1

 Procesando con temperature=0.3


In [33]:
df_qa.describe()

,chunk,question,answer,answer_modelo_rag_dslr_h_0.1,answer_modelo_rag_dslr_h_0.3
count,511,511,511,456,100
unique,511,511,511,456,100
top,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,Water can be classified based on Total Dissolv...,Water can be classified based on Total Dissolv...
freq,1,1,1,1,1


In [38]:
df_qa.head()

,chunk,question,answer,answer_modelo_rag_dslr_h_0.1,answer_modelo_rag_dslr_h_0.3
0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,Water can be classified based on Total Dissolv...,Water can be classified based on Total Dissolv...
1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...
2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,The maximum recovery value for membrane soften...,El contexto proporcionado no especifica direct...
3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,The average temperature is used for performanc...
4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,Scaling substances must be removed from treate...


In [50]:
df_qa.head(100).describe()

,chunk,question,answer,answer_modelo_rag_dslr_h_0.1,answer_modelo_rag_dslr_h_0.3
count,100,100,100,100,100
unique,100,100,100,100,100
top,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,Water can be classified based on Total Dissolv...,Water can be classified based on Total Dissolv...
freq,1,1,1,1,1


In [58]:
df_qa.to_csv("../dataQA/hiperparameters.csv")

### ROUGUE SCORE

In [ ]:
import os
output_dir = os.path.join("..", "resultados")
# El ROUGE score (Recall-Oriented Understudy for Gisting Evaluation) es una métrica ampliamente usada para evaluar la calidad de textos generados automáticamente
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
## Definimos funcion para hallar ROUGE
def calculate_rouge_score(column_name,data_name):
    # Evaluar ROUGE para cada par respuesta_modelo - respuesta_referencia
    rouge_scores = []
    
    for i, row in df_qa.head(100).iterrows():
        ref = row["answer"]  # respuesta de referencia
        gen = row[column_name]  # respuesta generada
        score = scorer.score(ref, gen)
        rouge_scores.append({
            "ROUGE-1": score["rouge1"].fmeasure,
            "ROUGE-2": score["rouge2"].fmeasure,
            "ROUGE-L": score["rougeL"].fmeasure
        })
        
    #Resultados
    # Convertir a DataFrame y unirlo al original
    df_rouge = pd.DataFrame(rouge_scores)
    df_resultado = pd.concat([df_qa.head(100).reset_index(drop=True), df_rouge], axis=1)

    # Mostrar puntajes promedio
    promedios = df_rouge.mean()
    print("🔍 Promedios ROUGE:")
    print(promedios.round(4))
    
    # Ver resultados por pregunta
    print("\n📌 Ejemplos con ROUGE:")
    print(df_resultado[["question", "ROUGE-1", "ROUGE-2", "ROUGE-L"]])

    # Guardar
    output_path = os.path.join(output_dir, data_name)
    df_resultado.to_csv(output_path, index=False)
    print(f"\n✅ Guardado en {data_name}")

In [46]:
calculate_rouge_score("answer_modelo_rag_dslr_h_0.1","evaluacion_dslr_con_rouge_h_1.csv")

🔍 Promedios ROUGE:
ROUGE-1    0.2109
ROUGE-2    0.0619
ROUGE-L    0.1580
dtype: float64

📌 Ejemplos con ROUGE:
                                              question   ROUGE-1   ROUGE-2  \
0    What types of water are classified based on To...  0.314815  0.075472   
1    Why is it important to limit product recovery ...  0.229508  0.116667   
2    How is the maximum recovery value determined f...  0.208791  0.077778   
3    Why is average temperature used for performanc...  0.296296  0.090226   
4    Why must scaling substances be removed from tr...  0.271186  0.068966   
..                                                 ...       ...       ...   
506  What are common causes for failed serial commu...       NaN       NaN   
507  How should you troubleshoot incorrect output b...       NaN       NaN   
508  How can you fix alarm and control mode issues ...       NaN       NaN   
509  What steps can correct poor control accuracy o...       NaN       NaN   
510  How should error messages 

In [47]:
calculate_rouge_score("answer_modelo_rag_dslr_h_0.3","evaluacion_dslr_con_rouge_h_3.csv")

🔍 Promedios ROUGE:
ROUGE-1    0.2036
ROUGE-2    0.0609
ROUGE-L    0.1545
dtype: float64

📌 Ejemplos con ROUGE:
                                              question   ROUGE-1   ROUGE-2  \
0    What types of water are classified based on To...  0.288136  0.068966   
1    Why is it important to limit product recovery ...  0.254237  0.103448   
2    How is the maximum recovery value determined f...  0.010471  0.000000   
3    Why is average temperature used for performanc...  0.303030  0.076923   
4    Why must scaling substances be removed from tr...  0.275862  0.070175   
..                                                 ...       ...       ...   
506  What are common causes for failed serial commu...       NaN       NaN   
507  How should you troubleshoot incorrect output b...       NaN       NaN   
508  How can you fix alarm and control mode issues ...       NaN       NaN   
509  What steps can correct poor control accuracy o...       NaN       NaN   
510  How should error messages 

### BERT SCORE

In [55]:
from bert_score import score
import pandas as pd
import os

def calculate_bert_score(name_column, name_data):
    refs = df_qa.head(100)["answer"].astype(str).tolist()
    gens = df_qa.head(100)[name_column].astype(str).tolist()

    # Calcular BERTScore para todo el batch
    P, R, F1 = score(gens, refs, lang="en", verbose=False)

    df_bert = pd.DataFrame({
        "PRECISION": P.tolist(),
        "RECALL": R.tolist(),
        "F1": F1.tolist()
    })

    df_resultado_bert = pd.concat([df_qa.head(100).reset_index(drop=True), df_bert], axis=1)

    # Mostrar promedio
    promedios = df_bert.mean()
    print("🔍 Promedios BERTSCORE:")
    print(promedios.round(4))

    print("\n📌 Ejemplos con BERT:")
    print(df_resultado_bert[["question", "PRECISION", "RECALL", "F1"]].head(10))

    # Guardar
    output_path = os.path.join(output_dir, name_data)
    df_resultado_bert.to_csv(output_path, index=False)

In [56]:
calculate_bert_score("answer_modelo_rag_dslr_h_0.1","evaluacion_dslr_con_bert_h_1.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8428
RECALL       0.8914
F1           0.8663
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.825854  0.879782   
1  Why is it important to limit product recovery ...   0.853246  0.887393   
2  How is the maximum recovery value determined f...   0.824857  0.910622   
3  Why is average temperature used for performanc...   0.851064  0.906721   
4  Why must scaling substances be removed from tr...   0.866750  0.904464   
5  What causes scaling due to supersaturation in ...   0.822966  0.889470   
6   How is the scaling tendency of salts quantified?   0.829273  0.838631   
7  What factors influence the effectiveness of an...   0.794169  0.850845   
8  What are the benefits of phosphate-based and p...   0.824742  0.877463   
9  Why must antiscalants be handled carefully in ...   0.834347  0.869429   

         F1  
0  0.851966  
1  0

In [57]:
calculate_bert_score("answer_modelo_rag_dslr_h_0.3","evaluacion_dslr_con_bert_h_3.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8405
RECALL       0.8905
F1           0.8646
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.816978  0.879155   
1  Why is it important to limit product recovery ...   0.854682  0.890459   
2  How is the maximum recovery value determined f...   0.734009  0.858188   
3  Why is average temperature used for performanc...   0.853528  0.907064   
4  Why must scaling substances be removed from tr...   0.865885  0.902733   
5  What causes scaling due to supersaturation in ...   0.821501  0.894822   
6   How is the scaling tendency of salts quantified?   0.816520  0.836578   
7  What factors influence the effectiveness of an...   0.801620  0.855427   
8  What are the benefits of phosphate-based and p...   0.825877  0.881260   
9  Why must antiscalants be handled carefully in ...   0.831237  0.868600   

         F1  
0  0.846927  
1  0

In [59]:
df_qa_gemma= pd.read_csv("../dataQA/df_qa_gemma.csv")
df_qa_gemma.head()

,chunk,question,answer,answer_modelo_rag,retrieved,answer_modelo,claims_phi4,answer_modelo_rag_dslr,retrieved_dslr
0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on Total Dissolved Solids (TDS) levels, ...","[{""titulo"": ""7.5 RO FOULING substance (anaysis...",Total Dissolved Solids (TDS) levels are used t...,"['Brackish Water TDS levels are between 1,000 ...",Water can be classified based on Total Dissolv...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in reverse osmosis (...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...",Limiting product recovery in Reverse Osmosis (...,['Limiting product recovery in RO systems is i...,Limiting product recovery in RO (Reverse Osmos...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,El valor máximo de recuperación para sistemas ...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...","Membrane softening systems, such as those used...",['El valor máximo de recuperación se determina...,La máxima recuperación permitida para sistemas...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...","The term ""RO membranes"" typically refers to re...",['The average temperature is used for performa...,The average temperature is used for performanc...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...","In reverse osmosis (RO) systems, scaling subst...",['Scaling substances must be removed from trea...,Scaling substances must be removed from treate...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."


In [60]:
df_qa["answer_modelo_rag_dslr_h_0.5"] = df_qa_gemma["answer_modelo_rag_dslr"]

In [61]:
calculate_rouge_score("answer_modelo_rag_dslr_h_0.5","evaluacion_dslr_con_rouge_h_5.csv")

🔍 Promedios ROUGE:
ROUGE-1    0.2058
ROUGE-2    0.0627
ROUGE-L    0.1573
dtype: float64

📌 Ejemplos con ROUGE:
                                              question   ROUGE-1   ROUGE-2  \
0    What types of water are classified based on To...  0.259542  0.062016   
1    Why is it important to limit product recovery ...  0.252252  0.128440   
2    How is the maximum recovery value determined f...  0.000000  0.000000   
3    Why is average temperature used for performanc...  0.272109  0.096552   
4    Why must scaling substances be removed from tr...  0.262295  0.050000   
..                                                 ...       ...       ...   
506  What are common causes for failed serial commu...       NaN       NaN   
507  How should you troubleshoot incorrect output b...       NaN       NaN   
508  How can you fix alarm and control mode issues ...       NaN       NaN   
509  What steps can correct poor control accuracy o...       NaN       NaN   
510  How should error messages 

In [62]:
calculate_bert_score("answer_modelo_rag_dslr_h_0.5","evaluacion_dslr_con_bert_h_5.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8404
RECALL       0.8908
F1           0.8647
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.816443  0.869528   
1  Why is it important to limit product recovery ...   0.858346  0.889183   
2  How is the maximum recovery value determined f...   0.721848  0.850407   
3  Why is average temperature used for performanc...   0.847332  0.909696   
4  Why must scaling substances be removed from tr...   0.857442  0.900266   
5  What causes scaling due to supersaturation in ...   0.820067  0.891562   
6   How is the scaling tendency of salts quantified?   0.836627  0.840376   
7  What factors influence the effectiveness of an...   0.796512  0.853372   
8  What are the benefits of phosphate-based and p...   0.842581  0.890864   
9  Why must antiscalants be handled carefully in ...   0.829654  0.874948   

         F1  
0  0.842149  
1  0

### Comparando

In [69]:
df_bert_h1 = pd.read_csv("../resultados/evaluacion_dslr_con_bert_h_1.csv")
df_bert_h3 = pd.read_csv("../resultados/evaluacion_dslr_con_bert_h_3.csv")
df_bert_h5 = pd.read_csv("../resultados/evaluacion_dslr_con_bert_h_5.csv")
df_bert_h1.head()

,chunk,question,answer,answer_modelo_rag_dslr_h_0.1,answer_modelo_rag_dslr_h_0.3,PRECISION,RECALL,F1
0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,Water can be classified based on Total Dissolv...,Water can be classified based on Total Dissolv...,0.825854,0.879782,0.851966
1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,0.853246,0.887393,0.869985
2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,The maximum recovery value for membrane soften...,El contexto proporcionado no especifica direct...,0.824857,0.910622,0.865620
3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,The average temperature is used for performanc...,0.851064,0.906721,0.878011
4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,0.866750,0.904464,0.885206


In [79]:
mean_BERT_1=[]
mean_BERT_3=[]
mean_BERT_5=[]

mean_BERT_1.append(df_bert_h1.head(100)["F1"].mean().round(4))
mean_BERT_3.append(df_bert_h3.head(100)["F1"].mean().round(4))
mean_BERT_5.append(df_bert_h5.head(100)["F1"].mean().round(4))

df_mean = pd.DataFrame({

    "BERT-H-1": mean_BERT_1,
    "BERT-H-3": mean_BERT_3,
    "BERT-H-5": mean_BERT_5,
})

In [80]:
df_mean

,BERT-H-1,BERT-H-3,BERT-H-5
0,0.8663,0.8646,0.8647


In [75]:
df_rouge_h1 = pd.read_csv("../resultados/evaluacion_dslr_con_rouge_h_1.csv")
df_rouge_h3 = pd.read_csv("../resultados/evaluacion_dslr_con_rouge_h_3.csv")
df_rouge_h5 = pd.read_csv("../resultados/evaluacion_dslr_con_rouge_h_5.csv")

In [77]:
mean_rouge_1=[]
mean_rouge_3=[]
mean_rouge_5=[]

mean_rouge_1.append(df_rouge_h1.head(100)["ROUGE-1"].mean().round(4))
mean_rouge_3.append(df_rouge_h3.head(100)["ROUGE-1"].mean().round(4))
mean_rouge_5.append(df_rouge_h5.head(100)["ROUGE-1"].mean().round(4))

df_mean = pd.DataFrame({

    "ROUGE-1-H-1": mean_rouge_1,
    "ROUGE-1-H-3": mean_rouge_3,
    "ROUGE-1-H-5": mean_rouge_5,
})

In [78]:
df_mean

,ROUGE-1-H-1,ROUGE-1-H-3,ROUGE-1-H-5
0,0.2109,0.2036,0.2058
